In [ ]:
# packages
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix


In [ ]:
# data prep
# source: https://www.kaggle.com/datasets/rashikrahmanpritom/heart-attack-analysis-prediction-dataset
df = pd.read_csv("heart.csv")
df.head()


In [ ]:
# separate independent / dependent features
X = np.array(df.loc[:, df.columns != "output"])
y = np.array(df["output"])

print(f"X: {X.shape}, y: {y.shape}")


In [ ]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)


In [ ]:
# scale the data
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)


In [ ]:
# network class
class NeuralNetworkFromScratch:
    def __init__(self, LR, X_train, y_train, X_test, y_test):
        self.w = np.random.randn(X_train_scale.shape[1])
        self.b = np.random.randn()
        self.LR = LR
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.L_train = []
        self.L_test = []

    def activation(self, x):
        return 1 / (1 + np.exp(-x))

    def dactivation(self, x):
        return self.activation(x) * (1 - self.activation(x))

    def forward(self, X):
        hidden_1 = np.dot(X, self.w) + self.b
        activate_1 = self.activation(hidden_1)
        return activate_1

    def backward(self, X, y_true):
        hidden_1 = np.dot(X, self.w) + self.b  # z
        y_pred = self.forward(X)
        dL_dpred = 2 * (y_pred - y_true)
        dpred_dhidden1 = self.dactivation(hidden_1)
        dhidden_db = 1
        dhidden_dw = X
        dL_db = dL_dpred * dpred_dhidden1 * dhidden_db
        dL_dw = dL_dpred * dpred_dhidden1 * dhidden_dw  # chain rule
        return dL_db, dL_dw

    def optimizer(self, dL_db, dL_dw):
        self.w -= dL_dw * self.LR
        self.b -= dL_db * self.LR

    def train(self, ITERATIONS=500):
        for i in range(ITERATIONS):

            random_pos = np.random.randint(len(self.X_train))
            
            # forward pass
            y_train_true = self.y_train[random_pos]
            y_train_pred = self.forward(self.X_train[random_pos])
            # y_train_true = self.X_train[pos]
            # y_train_pred = self.forward(self.X_train[pos])
            L = np.square(y_train_pred - y_train_true)
            self.L_train.append(L)
            # calc gradients
            dL_db, dL_dw = self.backward(
                self.X_train[random_pos], self.y_train[random_pos]
            )
            self.optimizer(dL_db, dL_dw)
            L_sum = 0
            y_test_pred = self.forward(self.X_test)
            L_sum = np.sum((y_test_pred - self.y_test) ** 2)
            self.L_test.append(L_sum)


In [ ]:
# Hyper parameters
LR = 0.1
ITERATIONS = 1000


In [ ]:
# model instance and training
nn = NeuralNetworkFromScratch(LR=LR, X_train=X_train_scale, y_train=y_train, X_test=X_test_scale, y_test=y_test)
nn.train(ITERATIONS=ITERATIONS)


In [ ]:
# check losses
sns.lineplot(x=list(range(len(nn.L_test))), y=nn.L_test)


In [ ]:
# iterate over test data
total = X_test_scale.shape[0]
correct = 0
y_preds = []
for i in range(total):
    y_true = y_test[i]
    y_pred = np.round(nn.forward(X_test_scale[i]))
    y_preds.append(y_pred)
    correct += 1 if y_true == y_pred else 0


In [ ]:
# Calculate Accuracy
acc = correct / total


In [ ]:
# Baseline Classifier
from collections import Counter
Counter(y_test)


In [ ]:
# Confusion Matrix
confusion_matrix(y_true = y_test, y_pred = y_preds)
